#  합성 가능성(Synthetic Feasibility)

합성 가능성은 대규모 분자 열거(enumeration)를 수행할 때 중요한 문제입니다. 열거된 분자들은 다른 화학적 성질이 컴퓨터상(in silico)으로 아무리 좋아 보여도, 실제로는 만들기가 매우 어려워서 살펴볼 가치가 없는 경우가 많습니다. 이번 튜토리얼에서는 [ScScore](https://pubs.acs.org/doi/abs/10.1021/acs.jcim.7b00622) 모델 [1]을 학습시키는 방법을 다룹니다.

이 모델의 핵심 아이디어는, 한 분자가 다른 분자보다 "더 복잡한" 분자 쌍(pair)으로 학습시키는 것입니다. 그러면 신경망은 이러한 분자 간 쌍대 순서(pairwise ordering)를 유지하려는 점수(score)를 만들어낼 수 있습니다. 최종 결과물은 분자의 상대적 복잡도(relative complexity)를 매겨주는 모델입니다.

원 논문은 Reaxys에 있는 모든 반응(reaction)으로 학습시키며, 생성물(product)이 반응물보다 더 복잡하다고 선언합니다. 하지만 이런 학습 세트는 비용이 너무 많이 들기 때문에, 우리는 대신 임의의 분자들로 학습하되 SMILES 문자열이 더 길면 더 복잡하다고 선언하겠습니다. 실제 현업에서는 프로젝트에 맞는 어떤 복잡도 척도든 사용할 수 있습니다.

이번 튜토리얼에서는 Tox21 데이터셋을 사용해 간단한 합성 가능성 모델을 학습시키겠습니다.

## Colab에서 실행하기

이 노트북은 Google Colab에서 실행하는 것을 권장합니다. 아래 Colab 배지를 클릭하면 바로 열립니다. (Kaggle에서도 열 수 있습니다.)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isg-yhlee93/lecture/blob/main/Molecular%20Machine%20Learning/4_Synthetic_Feasibility_Scoring.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https://github.com/isg-yhlee93/lecture/blob/main/Molecular%20Machine%20Learning/4_Synthetic_Feasibility_Scoring.ipynb)


In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # DeepChem은 Keras 2 기반이므로 레거시 Keras 사용 (Keras 3 비호환 회피)

!pip install -qq --pre deepchem tf_keras   # tf_keras = Keras 2 백엔드 제공
import deepchem

# 불필요한 경고·로그 메시지 끄기 (화면을 깔끔하게 보기 위함)
import warnings
warnings.filterwarnings('ignore')   # 파이썬 경고 숨기기

from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*')      # RDKit 로그 끄기

deepchem.__version__


# 데이터셋 만들기

먼저 다룰 분자들을 불러오는 것부터 시작하겠습니다. Tox21을 불러오되 `splitter=None`을 지정하여 모든 데이터가 하나의 데이터셋으로 반환되도록 합니다.


In [ ]:
import deepchem as dc
# Tox21을 원본(Raw) 형태로, 분할 없이(splitter=None) 불러옵니다.
tasks, datasets, transformers = dc.molnet.load_tox21(featurizer='Raw', splitter=None)
molecules = datasets[0].X


ScScore는 상대적 복잡도로 학습되기 때문에, 데이터셋의 `X` 텐서가 3개의 차원 `(sample_id, molecule_id, features)`을 갖기를 원합니다. 하나의 샘플이 두 분자의 쌍이므로 `molecule_id` 차원의 크기는 2입니다. 레이블(label)은 첫 번째 분자가 두 번째 분자보다 더 복잡하면 1입니다. 아래에서 소개하는 `create_dataset` 함수는 주어진 목록에서 무작위로 SMILES 문자열 쌍을 뽑아, 이 복잡도 척도에 따라 순위를 매깁니다.

실제 현업에서는 구매 비용이나 필요한 반응 단계 수 등을 복잡도 점수로 사용할 수 있습니다.


In [ ]:
from rdkit import Chem
import random
from deepchem.feat import CircularFingerprint
import numpy as np


def create_dataset(fingerprints, smiles_lens, ds_size=100000):
    """
    m1: np.Array의 목록
        분자들의 지문(fingerprint)
    m2: int의 목록
        분자 SMILES 문자열의 길이

    반환값:
        ScScore 모델 입력용 dc.data.Dataset

    Dataset.X
        형태(shape): (sample_id, molecule_id, features)
    Dataset.y
        형태(shape): (sample_id,)
        값: 0번 인덱스 분자가 더 복잡하면 1
            1번 인덱스 분자가 더 복잡하면 0
    """
    X, y = [], []
    all_data = list(zip(fingerprints, smiles_lens))
    while len(y) < ds_size:
        # 무작위로 두 분자를 고릅니다.
        i1 = random.randrange(0, len(smiles_lens))
        i2 = random.randrange(0, len(smiles_lens))
        m1 = all_data[i1]
        m2 = all_data[i2]
        if m1[1] == m2[1]:   # SMILES 길이가 같으면(복잡도 동률) 건너뜁니다.
            continue
        if m1[1] > m2[1]:    # 첫 번째 분자가 더 길면(더 복잡하면) 레이블 1
            y.append(1.0)
        else:
            y.append(0.0)
        X.append([m1[0], m2[0]])
    return dc.data.NumpyDataset(np.array(X), np.expand_dims(np.array(y), axis=1))


복잡도 순위 매기기가 준비되었으니 이제 데이터셋을 구성할 수 있습니다. 먼저 분자 목록을 학습 세트와 테스트 세트로 무작위 분할하는 것부터 시작하겠습니다.


In [ ]:
molecule_ds = dc.data.NumpyDataset(np.array(molecules))
# 무작위 분할기로 학습/테스트 세트를 나눕니다.
splitter = dc.splits.RandomSplitter()
train_mols, test_mols = splitter.train_test_split(molecule_ds)


모든 분자를 (원 논문과 동일하게) 카이랄성(chirality)을 포함한 ECFP 지문으로 특징화한 다음, 위에서 정의한 함수로 쌍대(pairwise) 데이터셋을 구성하겠습니다. 여기서는 원형 지문(Circular Fingerprint) 특징화 도구를 사용하며, 지문 크기 `n_features`, 지문 반경 `radius`, 카이랄성 고려 여부 `chiral` 같은 매개변수를 지정합니다. 원형 지문은 분자의 구조 정보를 인코딩하는 대표적인 분자 지문 종류입니다.


In [ ]:
n_features = 1024
# 카이랄성을 고려한 원형 지문 특징화 도구를 만듭니다.
featurizer = dc.feat.CircularFingerprint(size=n_features, radius=2, chiral=True)
train_features = featurizer.featurize(train_mols.X)
# 각 분자의 SMILES 길이를 복잡도 척도로 사용합니다.
train_smiles_len = [len(Chem.MolToSmiles(x)) for x in train_mols.X]
train_dataset = create_dataset(train_features, train_smiles_len)


이제 데이터셋이 만들어졌으니, 이 데이터셋으로 `ScScoreModel`을 학습시켜 봅시다. (학습은 Colab T4 GPU 기준 약 1분 정도 걸립니다.)


In [ ]:
model = dc.models.ScScoreModel(n_features=n_features)
model.fit(train_dataset, nb_epoch=20)   # Colab T4 GPU 기준 약 1분 소요


# 모델 성능

홀드아웃(holdout) 분자들에 대해 모델이 얼마나 잘 동작하는지 평가해 봅시다. ScScore는 한 번도 본 적 없는 분자들의 SMILES 문자열 길이를 잘 추적해야 합니다.


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline


In [ ]:
# 테스트 분자들의 ScScore와 SMILES 길이를 계산합니다.
mol_scores = model.predict_mols(test_mols.X)
smiles_lengths = [len(Chem.MolToSmiles(x)) for x in test_mols.X]


이제 matplotlib을 사용해 분자의 SMILES 문자열 길이와 ScScore를 함께 그려 보겠습니다.


In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(smiles_lengths, mol_scores)
plt.xlim(0,80)
plt.xlabel("SMILES 길이")
plt.ylabel("ScScore")
plt.show()


그래프에서 볼 수 있듯이, 모델은 대체로 SMILES 길이를 잘 추적합니다. 8~30자 구간에서 좋은 변별력(enrichment)을 보이며, 아주 짧거나 아주 긴 SMILES 문자열 양 극단도 정확히 맞춥니다.

이제 여러분은 SMILES 길이보다 더 의미 있는 척도로 직접 모델을 학습시킬 수 있습니다!


### 잠깐 — ScScore는 어떻게 "분자 1개"에 점수를 매길까?

학습 데이터는 분자 **쌍(pair)**과 0/1 레이블("어느 쪽이 더 복잡한가")이지만, 모델 자체는 **분자 1개 → 점수 1개(1~5)** 함수입니다. 쌍 비교는 **모델 구조가 아니라 손실(loss) 계산에서만** 일어납니다.

- **모델이 예측하는 것:** 마지막 층은 **출력 노드가 1개**뿐이라, 분자 하나당 **스칼라 점수 하나**를 냅니다. 이 raw 값을 시그모이드로 **[1, 5]** 범위에 묶습니다 (1=단순, 5=복잡). 즉 분류가 아니라 "복잡도 점수"라는 스칼라를 예측합니다.
- **순전파(forward):** 같은 신경망을 두 분자에 각각 적용해 `score1 = f(mol1)`, `score2 = f(mol2)`를 구합니다 (가중치를 공유하는 샴(Siamese) 구조).
- **손실(loss):** 분자별 "정답 점수"는 없습니다. 대신 **두 점수의 차이 `score1 - score2`** 를 보고, "더 복잡한 쪽(레이블 1)이 더 높은 점수를 받도록" 가중치를 갱신합니다. 즉 절대값을 맞추는 게 아니라 **대소 관계(순위)를 맞추는** 손실입니다 — 이런 방식을 **순위 학습(learning to rank)**이라고 합니다.
- **절대 점수는 어디서?** (1) 출력이 [1, 5]로 묶여 스케일이 고정되고, (2) 수많은 겹치는 쌍의 순서를 동시에 만족시키다 보면 분자마다 일관된 절대값이 정해집니다. 그래서 학습 후엔 분자 하나만 넣어도 점수가 나오고(`predict_mols`), 아주 단순/복잡한 분자는 양 끝(1·5)에 포화됩니다.

> 비유: 체스 **Elo 점수**와 같습니다. 관측되는 건 "A가 B를 이김"이라는 쌍대 승패뿐이고 정답 등급은 없지만, 수많은 대결이 맞물려 선수마다 **절대 등급**이 정해지죠. ScScore도 쌍대 비교만으로 분자마다 절대 복잡도 점수를 학습합니다.


# 참고 문헌(Bibliography)

[1] https://pubs.acs.org/doi/abs/10.1021/acs.jcim.7b00622
